# Lab 6 — Segmentation Summary

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

Translate cluster IDs into segment-level analytics and business narrative.

**Dataset:** `data/nyse/nyse_stocks.csv` (500 rows, 25 symbols)

## Setup and final segmentation assignment

In [ ]:
%matplotlib inline

from pathlib import Path

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

X_scaled = StandardScaler().fit_transform(features[FEATURE_COLUMNS])

k = 4
features = features.copy()
features["segment"] = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)

print("Lab 6 — Segmentation summary")
print(f"segments (k): {k}")


## Segment size counts

In [ ]:
segment_sizes = features["segment"].value_counts().sort_index()
segment_sizes_dict = segment_sizes.to_dict()

print(f"symbols per segment: {segment_sizes_dict}")
display(segment_sizes.rename("count").to_frame())


## Segment feature means

In [ ]:
summary = (
    features.groupby("segment")[FEATURE_COLUMNS]
    .mean()
    .round(2)
    .reset_index()
)

print("segment means:")
display(summary)


## Sample symbols per segment

In [ ]:
print("sample symbols per segment:")
sample_rows = []
for seg in sorted(features["segment"].unique()):
    symbols = features.loc[features["segment"] == seg, "symbol"].head(4).tolist()
    print(f"  segment {seg}: {symbols}")
    sample_rows.append({"segment": seg, "sample_symbols": ", ".join(symbols)})

display(pd.DataFrame(sample_rows))


## Human-readable segment labels

In [ ]:
labels = {
    0: "high price / large-cap growth",
    1: "mid price / diversified large names",
    2: "lower price / higher volatility",
    3: "singleton — high price, low vol (PEP)",
}

summary_labeled = summary.copy()
summary_labeled["label"] = summary_labeled["segment"].map(labels)
display(summary_labeled)


## Symbol-level segment table

In [ ]:
display(
    features[["symbol", "segment", "avg_close", "volatility"]]
    .sort_values(["segment", "symbol"])
    .round(2)
)


## Segment size bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(
    x=segment_sizes.index.astype(str),
    y=segment_sizes.values,
    ax=ax,
    palette="Set2",
)
ax.set_xlabel("segment")
ax.set_ylabel("symbol count")
ax.set_title("NYSE K-Means segment sizes (k=4)")
plt.tight_layout()
plt.show()


## Day recap table

In [ ]:
recap = pd.DataFrame({
    "lab": [
        "1 K-Means",
        "2 Elbow",
        "3 DBSCAN",
        "4 Metrics",
        "5 Multi-view",
        "6 Summary",
    ],
    "checkpoint": [
        "inertia ≈ 45.86",
        "best k = 3",
        "3 clusters, 11 noise",
        "silhouette ≈ 0.24",
        "multi_cluster_view.png",
        str(segment_sizes_dict),
    ],
})
display(recap)


### Segmentation prompt 1

Sort segments by avg_close descending.

In [ ]:
display(summary.sort_values('avg_close', ascending=False))

### Segmentation prompt 2

Sort segments by volatility descending.

In [ ]:
display(summary.sort_values('volatility', ascending=False))

### Segmentation prompt 3

Compute spread between highest and lowest avg_close segment.

In [ ]:
print(round(summary['avg_close'].max() - summary['avg_close'].min(), 2))

### Segmentation prompt 4

List symbols in segment 0.

In [ ]:
print(features.loc[features['segment']==0, 'symbol'].tolist())

### Segmentation prompt 5

List symbols in segment 1.

In [ ]:
print(features.loc[features['segment']==1, 'symbol'].tolist())

### Segmentation prompt 6

List symbols in segment 2.

In [ ]:
print(features.loc[features['segment']==2, 'symbol'].tolist())

### Segmentation prompt 7

List symbols in segment 3.

In [ ]:
print(features.loc[features['segment']==3, 'symbol'].tolist())

### Segmentation prompt 8

Compute avg volume by segment.

In [ ]:
display(features.groupby('segment')['avg_volume'].mean().round(2))

### Segmentation prompt 9

Compute avg range by segment.

In [ ]:
display(features.groupby('segment')['avg_range'].mean().round(2))

### Segmentation prompt 10

Display segment medians.

In [ ]:
display(features.groupby('segment')[FEATURE_COLUMNS].median().round(2))

### Segmentation prompt 11

Count symbols per label text.

In [ ]:
tmp = features.copy(); tmp['label']=tmp['segment'].map(labels); print(tmp['label'].value_counts().to_dict())

### Segmentation prompt 12

Identify top 2 avg_close symbols in each segment.

In [ ]:
display(features.sort_values(['segment','avg_close'], ascending=[True, False]).groupby('segment').head(2)[['symbol','segment','avg_close']].round(2))

### Segmentation prompt 13

Check whether segment 3 is singleton.

In [ ]:
print(int((features['segment']==3).sum()))

### Segmentation prompt 14

Compute z-like spread on avg_close within segment 0.

In [ ]:
s0 = features.loc[features['segment']==0, 'avg_close']; print(round(float(s0.std(ddof=0)), 2))

### Segmentation prompt 15

Cross-check segment sizes dictionary.

In [ ]:
print(segment_sizes_dict)

### Segmentation prompt 16

Add narrative sentence for segment 0.

In [ ]:
print('Segment 0 contains high-price large-cap names with stronger average close.')

### Segmentation prompt 17

Add narrative sentence for segment 2.

In [ ]:
print('Segment 2 captures lower-priced names with relatively higher volatility.')

### Segmentation prompt 18

Compare segment means table with labeled table.

In [ ]:
display(summary_labeled[['segment','label','avg_close','volatility']])

### Segmentation prompt 19

Bridge to anomaly day.

In [ ]:
print('Next day uses imbalance and anomaly signals where cluster outliers can help feature design.')

### Segmentation prompt 20

Verify symbol uniqueness.

In [ ]:
print(features['symbol'].nunique(), len(features))

### Lab 6 quick recap 1

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 6 recap step 1: completed")

### Lab 6 quick recap 2

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 6 recap step 2: completed")

## Final checkpoint

In [ ]:
assert k == 4
assert segment_sizes_dict == {0: 8, 1: 9, 2: 7, 3: 1}
assert summary.loc[summary["segment"] == 0, "avg_close"].iloc[0] > 200
seg0_symbols = features.loc[features["segment"] == 0, "symbol"].head(4).tolist()
assert seg0_symbols == ["CSCO", "DIS", "MSFT", "NFLX"]
print("Numbers match — you're good.")



## Reflection

How would you rename segments for a portfolio manager audience?